# 03 Calibration And CI

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=check)

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    _run_setup('git fetch origin')
    current_branch = subprocess.check_output(
        'git branch --show-current',
        shell=True,
        text=True,
    ).strip()
    if current_branch != BRANCH:
        _run_setup(f'git checkout {BRANCH}')
    _run_setup(f'git pull --ff-only --autostash origin {BRANCH}')
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        _run_setup('git fetch origin')
        _run_setup(f'git pull --ff-only --autostash origin {BRANCH}', check=False)
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

def run(cmd, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, check=check)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Install Metric Dependencies

In [ ]:
INSTALL_METRIC_DEPS = True
if INSTALL_METRIC_DEPS:
    import importlib.util
    import subprocess
    import sys

    required = {
        'numpy': 'numpy',
        'pandas': 'pandas',
        'scipy': 'scipy',
        'sklearn': 'scikit-learn',
        'threadpoolctl': 'threadpoolctl',
        'matplotlib': 'matplotlib',
        'dateutil': 'python-dateutil',
    }
    missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
    print('Python:', sys.version)
    if missing:
        print('Installing missing metric dependencies:', missing)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    else:
        print('Metric dependencies already available; no pip install needed.')

    import matplotlib
    import numpy as np
    import scipy
    import sklearn
    print('numpy     :', np.__version__)
    print('scipy     :', scipy.__version__)
    print('sklearn   :', sklearn.__version__)
    print('matplotlib:', matplotlib.__version__)
else:
    print('Skipping metric dependency install.')


## Run Calibration And Bootstrap CI

In [ ]:
import os
import numpy as np

N_BOOT = 1000
N_BINS = 15
THRESHOLD = 0.5
OVERWRITE_METRICS = True

repo_dir = Path(globals().get('REPO_DIR', '/content/drive/MyDrive/ECG-Ramba/ECG-RAMBA'))
if repo_dir.exists() and (repo_dir / 'configs' / 'config.py').exists() and Path.cwd() != repo_dir:
    os.chdir(repo_dir)

print('cwd:', Path.cwd())

candidate_pred_dirs = []
for path in [
    Path('reports/revision/predictions'),
    repo_dir / 'reports' / 'revision' / 'predictions',
    Path('/content/drive/MyDrive/ECG-Ramba/ECG-RAMBA/reports/revision/predictions'),
]:
    if path not in candidate_pred_dirs:
        candidate_pred_dirs.append(path)

metric_dir = Path('reports/revision/metrics')
metric_dir.mkdir(parents=True, exist_ok=True)

metric_prediction_files = []
skipped_prediction_files = []
seen_predictions = set()

print('Prediction directories checked:')
for pred_dir in candidate_pred_dirs:
    files = sorted(pred_dir.glob('*.npz')) if pred_dir.exists() else []
    print(f' - {pred_dir} | exists={pred_dir.exists()} | npz={len(files)}')
    for pred in files:
        pred_key = str(pred.resolve()) if pred.exists() else str(pred)
        if pred_key in seen_predictions:
            continue
        seen_predictions.add(pred_key)
        try:
            with np.load(pred, allow_pickle=True) as data:
                keys = set(data.files)
                if {'y_true', 'y_prob'}.issubset(keys):
                    metric_prediction_files.append(pred)
                else:
                    skipped_prediction_files.append((pred, sorted(keys)))
        except Exception as exc:
            skipped_prediction_files.append((pred, [f'load_error={type(exc).__name__}: {exc}']))

if skipped_prediction_files:
    print('Skipping NPZ files that are not record-level metric predictions:')
    for path, keys in skipped_prediction_files:
        print(' -', path, 'keys=', keys)

if not metric_prediction_files:
    revision_root = repo_dir / 'reports' / 'revision'
    nearby = sorted(revision_root.glob('**/*.npz')) if revision_root.exists() else []
    if nearby:
        print('Nearby NPZ files under revision root:')
        for path in nearby[:30]:
            print(' -', path)
    raise FileNotFoundError(
        'No metric-ready prediction NPZ files found. Expected a file such as '
        f'{repo_dir / "reports" / "revision" / "predictions" / "oof_full_predictions.npz"}. '
        'Run notebook 02 first, or run notebook 03 from the setup cell so cwd points to the Drive repo.'
    )

print('Metric-ready prediction files:')
for pred in metric_prediction_files:
    print(' -', pred)

for pred in metric_prediction_files:
    out = metric_dir / f'calibration_ci_{pred.stem}.json'
    if out.exists() and not OVERWRITE_METRICS:
        print(f'SKIP existing: {out}')
        continue
    cmd = (
        f'python scripts/revision/04_calibration_ci.py '
        f'--predictions {pred} --out {out} '
        f'--threshold {THRESHOLD} --n-bins {N_BINS} --n-boot {N_BOOT}'
    )
    run(cmd)


## Summarize Metric Files

In [ ]:
metric_jsons = sorted(Path('reports/revision/metrics').glob('calibration_ci_*.json'))
if not metric_jsons:
    raise FileNotFoundError('No calibration_ci_*.json files found.')

for path in metric_jsons:
    payload = json.loads(path.read_text(encoding='utf-8'))
    print('\n' + str(path))
    print('dataset:', payload.get('dataset'))
    print('shape:', payload.get('shape'))
    print('metrics:', payload.get('metrics'))
    print('calibration:', payload.get('calibration'))
    print('bootstrap_ci:', payload.get('bootstrap_ci'))
    print('artifacts:', payload.get('artifacts'))


## Build Reviewer Tables

In [ ]:
import pandas as pd
from IPython.display import display

table_dir = Path('reports/revision/tables')
table_dir.mkdir(parents=True, exist_ok=True)

calibration_rows = []
bootstrap_rows = []

for path in sorted(Path('reports/revision/metrics').glob('calibration_ci_*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    metrics = payload.get('metrics', {})
    calibration = payload.get('calibration', {})
    shape = payload.get('shape', {})
    row = {
        'dataset': payload.get('dataset') or Path(payload.get('predictions', path.stem)).stem,
        'predictions': payload.get('predictions'),
        'protocol': payload.get('protocol'),
        'git_commit': payload.get('git_commit'),
        'n_records': shape.get('y_true', [None, None])[0],
        'n_classes': shape.get('y_true', [None, None])[1],
        'threshold': payload.get('threshold'),
        'n_bins': payload.get('n_bins'),
        'n_boot': payload.get('n_boot'),
    }
    row.update(metrics)
    row.update(calibration)
    calibration_rows.append(row)

    for metric_name, ci in payload.get('bootstrap_ci', {}).items():
        bootstrap_rows.append({
            'dataset': row['dataset'],
            'metric': metric_name,
            'mean': ci.get('mean'),
            'ci_low': ci.get('lo'),
            'ci_high': ci.get('hi'),
            'n_boot_valid': ci.get('n_boot_valid'),
            'predictions': row['predictions'],
            'git_commit': row['git_commit'],
        })

table_calibration = table_dir / 'table_calibration.csv'
table_bootstrap = table_dir / 'table_bootstrap_ci.csv'
pd.DataFrame(calibration_rows).to_csv(table_calibration, index=False)
pd.DataFrame(bootstrap_rows).to_csv(table_bootstrap, index=False)

print('Wrote:', table_calibration)
print('Wrote:', table_bootstrap)
print('\nCalibration table preview:')
display(pd.DataFrame(calibration_rows))
print('\nBootstrap CI table preview:')
display(pd.DataFrame(bootstrap_rows))


In [ ]:
!python scripts/revision/05_artifact_inventory.py
